> **💾 First: save your own copy** · Unless this notebook is already in your own Google Drive, Colab does not keep your changes. Go to **File → Save a copy in Drive** and work in the copy that opens in a new tab. You find it later in Google Drive, in *Colab Notebooks*.

![Cultural Data Analysis](https://goto4711.github.io/cda-teaching-tools/assets/notebooks/week4-banner.png)

# Template: research with language models

This notebook is a starting point for a group project that uses language models, either as the **object** of your research (what do models say about X?) or as a **method** (a model codes your texts). It runs the workflow of the Week 4 group exercise on your own prompts and data.

**Prompting as an audit.** A common way to study language models as cultural objects is an **audit**: you send a model many prompts that are the same except for one thing, e.g. the group a sentence is about, the language, or the persona the model should take on, and compare the answers. Differences that follow that one change show what the model has learned to associate with it. Answers vary from run to run, so an audit needs many prompts and runs, a clear measure, and a record of every model and setting. An example from the course's teaching team: Mae Sosto, Delfina Sol Martinez Pandiani and Laura Hollink, [QueerGen: How LLMs Reflect Societal Norms on Gender and Sexuality in Sentence Completion Tasks](https://aclanthology.org/2026.findings-eacl.225.pdf) (Findings of EACL 2026). They built 3,100 sentences from templates, about people marked as queer, as not queer, or not marked at all, asked 14 models to complete them, and compared the sentiment, regard and toxicity of the completions. The Week 4 group exercise is a small audit too: the same prompt for five models in three languages.

The steps, with the functions of `llm_tools.py`:

| Step | Function |
|---|---|
| 0. Choose the models, also models from elsewhere | `openrouter_models()`, `has_key()`, `ask(..., base_url=...)` |
| 1. Ask one prompt | `ask()` |
| 2. Collect many answers, with their details | `collect()` |
| 3. Read the answers | `parse_dates()`, `parse_json()`, or your own function |
| 4. Code texts with a model | `coding_prompt()`, `code_with_llm()` |
| 5. Check the coding by hand | `validation_sample()`, `agreement()` |
| 6. Compare | `shares()`, `within_between()`, `jaccard_matrix()`, `brier()` |
| 7. Report what you did | a summary of models, settings, dates and costs |

Change the settings in each step for your project and delete what you don't need. Every function explains itself: run e.g. `help(collect)`.

**Without a key**, the notebook runs on the data of the Week 4 exercise, so you can try it first. With a key, it sends your own requests.

> **⚠️ Important** · **Keys:** put them in Colab's secrets (the key symbol on the left → *Add new secret*, switch on *Notebook access*), never into your code. `GEMINI_API_KEY` is free at [aistudio.google.com](https://aistudio.google.com) and works for models whose name starts with `gemini`. `OPENROUTER_API_KEY` ([openrouter.ai](https://openrouter.ai), paid) gives access to models from OpenAI, Anthropic, Mistral, DeepSeek and others, with names like `openai/gpt-4.1-mini`.
>
> **Data:** read the terms of the free tier: Google may use what you send to improve its products. Never send personal data or texts you are not allowed to share.
>
> **Costs:** `collect()` prints how many requests it will send. Try a small run first, and look at the tokens in the results before you scale up.

In [ ]:
import os, json, urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

if not os.path.exists('llm_tools.py'):   # keep the same version for the whole project
  urllib.request.urlretrieve('https://raw.githubusercontent.com/goto4711/cdai/main/llm_tools.py', 'llm_tools.py')
from llm_tools import *

print('llm_tools is ready.')

---

## 0. Choose the models

A name that starts with `gemini` goes straight to Google, with the free `GEMINI_API_KEY`. A name with a `/`, like `openai/gpt-4.1-mini`, goes through [OpenRouter](https://openrouter.ai), with `OPENROUTER_API_KEY`. OpenRouter offers hundreds of models from many companies through one API, with one key and one bill, so comparing models only means adding names to the list.

Two things to know about OpenRouter. Open models (Mistral, DeepSeek, Llama …) are run by several companies, and OpenRouter picks one for each request: `collect()` records which one in the column `served_by`. And prices and data policies differ between these companies: check them before you send anything sensitive.

`openrouter_models()` lists the current names and prices (in dollars per million tokens). It needs no key. Model names change every few months, so look them up here rather than copying them from old code.

In [ ]:
try:
  display(openrouter_models('mistral').head(10))       # e.g. 'gpt', 'claude', 'deepseek', 'llama'
except Exception as e:
  print('Could not reach OpenRouter:', e)

In [ ]:
MODELS = ['gemini-3.5-flash-lite']    # e.g. add 'openai/gpt-4.1-mini', 'anthropic/claude-haiku-4.5' with an OpenRouter key

for m in MODELS:
  print(f'{m:35}', 'key found' if has_key(m) else f'no key: add {key_name(m)} to the secrets')
MODELS = [m for m in MODELS if has_key(m)]
MODEL = MODELS[0] if MODELS else 'gemini-3.5-flash-lite'
DEMO = not MODELS
print('\nNo key: the notebook runs on the Week 4 data.' if DEMO else f'\nThe notebook uses: {MODELS}')

### Models from elsewhere

Most well-known models come from a few companies in the US and China. For an audit, it can be revealing to add models that were built in or for a particular language or region, and ask whether they answer differently. A student in an earlier year of this course compared an Arabic model from the United Arab Emirates with general-purpose models, to see whether it reproduces fewer prejudices about Arab women. Other examples: Arabic models such as Jais, Falcon-Arabic, ALLaM or Fanar; Chinese models such as Qwen or DeepSeek; SEA-LION for Southeast Asia; Sarvam for Indian languages; EuroLLM for European languages. Which of them you can reach changes often, so check before you plan.

There are three routes:

1. **OpenRouter**, if it offers the model: `openrouter_models('qwen')` shows it. Then use the name as in any other request.
2. **Another API in OpenAI's format**, e.g. the provider's own API, or Hugging Face's hosted models: `ask(prompt, model=..., base_url=..., key=...)`, where `key` is the name of the secret that holds the key. The cell below shows an example. Hugging Face only hosts some models, and it charges for requests (a free account has no credit).
3. **Run the model yourself in Colab.** Most of these models are on [Hugging Face](https://huggingface.co/models), and you have used Hugging Face models in the Week 4 workshop. But large chat models are much harder to run than the small models there: they often do not fit on Colab's free GPU, need special settings, and fail with errors that are hard to read. Choose this route only if the other two do not work, and talk to your project supervisor first.

> **⚠️ Important** · **A model from a region is not the region's voice.**
> * **Who built it, and why?** Many of these models come from companies or institutes close to a government, with their own goals. Much of their training data is often still English, or translated.
> * **What does "better" mean?** If a model reproduces fewer prejudices, it may know more, have stricter safety training, or follow its makers' norms about what may be said. Design the audit so you can tell these apart, e.g. by asking the same questions in several languages and to several models.
> * **Who checks?** Prompts and codes in a language need checking by someone who knows the language and its context.

In [ ]:
# Example: a model through another API in OpenAI's format. Change the address, the model name and the key's name.
# Hugging Face's hosted models: base_url='https://router.huggingface.co/v1', key='HF_TOKEN' (a Hugging Face token in the secrets)
OTHER = {'model': 'Qwen/Qwen3-8B', 'base_url': 'https://router.huggingface.co/v1', 'key': 'HF_TOKEN'}

if get_key(OTHER['key']):
  try:
    print(ask('Name three important dates in world history.', **OTHER))
  except Exception as e:
    print('The request did not work:', e)
else:
  print(f"No {OTHER['key']} in the secrets: skipped.")

---

## 1. Ask one prompt

Try your prompt by hand first, and read the answer. Change the prompt until the answer has a format you can read with code (step 3): e.g. one item per line, or JSON.

In [ ]:
PROMPT = '''Give me a list of 20 important dates in world history.
Do not aim for an academic compendium — just your own selection.
Format strictly as: YEAR | EVENT DESCRIPTION
One entry per line. No extra text.'''

if not DEMO:
  print(ask(PROMPT, model=MODEL))

---

## 2. Collect many answers

One answer is one sample: ask each prompt several times (`RUNS`), and vary one thing at a time, e.g. the model, the language or the wording of the prompt. `collect()` saves every answer at once in a CSV file, with the date, the model, the settings and the tokens. If it stops (e.g. the free tier allows only a few requests per minute), run the cell again: it goes on where it stopped.

Some models "think" before they answer, and the thinking is billed as output. Through OpenRouter, `collect(..., extra={'reasoning': {'effort': 'low'}})` keeps it short; `extra` is added to every request of that call, so call `collect()` once per model if the models need different settings (they all go into the same file).

Files in Colab disappear when the session ends. To keep them, connect your Google Drive (folder symbol on the left → *Mount Drive*) and save the file there, e.g. `FILE = '/content/drive/MyDrive/project/answers.csv'`.

In [ ]:
PROMPTS = {                          # name: prompt. The name is saved with every answer.
  'en': PROMPT,
}
RUNS = 3
TEMPERATURE = 1.0
FILE = 'answers.csv'

if DEMO:   # the Week 4 data in the same format
  if not os.path.exists('llm_dates_2026.json'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/goto4711/cdai/main/llm_dates_2026.json', 'llm_dates_2026.json')
  week4 = json.load(open('llm_dates_2026.json', encoding='utf-8'))
  answers = pd.DataFrame(week4['responses']).rename(columns={'language': 'prompt_id', 'input_tokens': 'tokens_in',
                                                           'output_tokens': 'tokens_out'})
  answers['model'] = answers['label']
  answers['date'] = week4['about']['collected']
else:
  answers = collect(PROMPTS, models=MODELS, runs=RUNS, temperature=TEMPERATURE, file=FILE)

answers[['model', 'prompt_id', 'run', 'text', 'tokens_in', 'tokens_out']].head()

---

## 3. Read the answers

Turn each answer into a table with **one row per item**. `parse_dates()` reads lists in the format `YEAR | EVENT`. For answers in JSON, use `parse_json()`. For another format, write your own function that turns one answer into a list.

Look at the answers that gave fewer items than you asked for: did the model ignore the format, or does your function miss something? Report such failures.

In [ ]:
answers['items'] = answers['text'].apply(parse_dates)
print('Items per answer:', answers['items'].apply(len).value_counts().sort_index().to_dict())

items = answers.explode('items').dropna(subset=['items'])
items[['year', 'item']] = pd.DataFrame(items['items'].tolist(), index=items.index)
items = items[['model', 'prompt_id', 'run', 'year', 'item']].reset_index(drop=True)
items.head()

---

## 4. Code the items with a model

Write down your categories and how to apply them, as you would for a human coder. `coding_prompt()` shows what the model receives; `code_with_llm()` sends it, 40 items per request, at temperature 0. Each distinct item is coded once. Codes that are not in your list become `unclear`.

Test the prompt on a few items first (e.g. `items['item'].head(20)`), read the codes, and improve the instructions before you code everything.

In [ ]:
INSTRUCTIONS = '''You code events from lists of "important dates in world history".
The region is where the event mainly took place. Russia counts as Europe.
Use "Global" only for events that happened in many world regions at once.'''
CATEGORIES = {
  'region': ['Europe', 'North America', 'Latin America', 'Africa', 'Middle East', 'South Asia', 'East Asia',
             'Southeast Asia & Oceania', 'Global'],
}
print(coding_prompt(INSTRUCTIONS, CATEGORIES))

In [ ]:
if DEMO:   # the Week 4 events were coded with a similar prompt
  codes = pd.DataFrame(week4['events']).rename(columns={'event': 'text'}).drop_duplicates('text')[['text', 'region']]
else:
  codes = code_with_llm(items['item'], INSTRUCTIONS, CATEGORIES, model=MODEL, file='codes.csv')

items = items.merge(codes[['text', *CATEGORIES]], left_on='item', right_on='text', how='left').drop(columns='text')
items.head()

---

## 5. Check the coding by hand

Draw a sample, and code it yourselves **before** you look at the model's codes: each of you alone, then compare among yourselves, then with the model. Report the agreement. If it is low, improve the categories or the instructions and code again.

`agreement()` gives the share of identical codes and Cohen's kappa (the agreement beyond chance: above 0.8 is very good, 0.6 to 0.8 good, below 0.4 weak). A sample of 50 to 100 items is common.

In [ ]:
sample = validation_sample(codes, n=20)

In [ ]:
# Your codes for the sample, in the same order, spelled exactly as in CATEGORIES
my_codes = []

In [ ]:
if len(my_codes) == len(sample):
  print(agreement(my_codes, sample['region']))
  check = sample.assign(you=my_codes)
  display(check[check['you'] != check['region']])      # where you and the model disagree
else:
  print(f'Code the {len(sample)} items of the sample first ({len(my_codes)} so far).')

---

## 6. Compare

Some ways to compare, from the Week 4 exercise:

* `shares(df, rows, columns)`: e.g. the share of each region per model or per prompt;
* `within_between(sets, labels)`: are the answers of one model more alike than those of different models? (Jaccard index);
* `jaccard_matrix(sets, labels)`: the same for every pair, as a table for a heatmap;
* `brier(probability, outcome)`: for probability forecasts.

In [ ]:
region_shares = shares(items, 'model', 'region')
region_shares.plot(kind='barh', stacked=True, figsize=(10, 4), colormap='tab10')
plt.legend(bbox_to_anchor=(1, 1))
plt.show()

In [ ]:
sets = items.groupby(['model', 'prompt_id', 'run'])['year'].apply(set).reset_index()
within, between = within_between(sets['year'], sets['model'])
print(f'Same model: {within:.2f}   Different models: {between:.2f}')

---

## 7. Report what you did

Others must be able to repeat your study. Report at least: the models (exact names) and when you asked them; the prompts and settings (temperature, number of runs); how many answers failed or had the wrong format; the coding prompt and the agreement with your hand coding. Through OpenRouter, also report which companies ran the models (`served_by`). The next cell summarises what `collect()` saved. Keep the CSV files with your project.

In [ ]:
summary = answers.groupby('model').agg(answers=('text', 'size'), first=('date', 'min'), last=('date', 'max'),
                                       tokens_in=('tokens_in', 'sum'), tokens_out=('tokens_out', 'sum'),
                                       served_by=('served_by', lambda s: ', '.join(sorted(set(map(str, s))))))
summary